In [ ]:
!uv add langgraph langchain langchain-openai langchain-community google-generativeai langchain-google-genai supabase python-dotenv pypdf docx2txt serpapi google-search-results

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("SUPABASE_URL:", os.getenv("SUPABASE_URL"))
print("SUPABASE_KEY:", os.getenv("SUPABASE_KEY"))
print("GEMINI_API_KEY:", os.getenv("GEMINI_API_KEY"))
print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY"))
print("SERPAPI_KEY:", os.getenv("SERPAPI_KEY"))

In [ ]:
from supabase import create_client

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

print(supabase.table("documents").select("count").execute())

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

for m in genai.list_models():
    if "embed" in m.name.lower():
        print(m.name)

In [ ]:
response = genai.embed_content(
    model="models/gemini-embedding-2",
    content="test sentence for pune domain",
    task_type="retrieval_document",
    output_dimensionality=1536
)

print("Embedding dimension:", len(response["embedding"]))

In [ ]:
import time

def ingest_domain(domain: str):
    domain_path = Path("data") / domain
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

    for file_path in domain_path.rglob("*.*"):
        docs = load_file(str(file_path))
        if not docs:
            continue

        chunks = splitter.split_documents(docs)
        texts = [c.page_content for c in chunks]
        total = len(chunks)

        print(f"\n📄 File: {file_path.name} — {total} chunks found")

        for i, text in enumerate(texts):
            print(f"   ⏳ Processing chunk {i+1} of {total}...", end="\r")
            
            response = genai.embed_content(
                model="models/gemini-embedding-2",
                content=text,
                task_type="retrieval_document",
                output_dimensionality=1536
            )
            supabase.table("documents").insert({
                "domain": domain,
                "source": file_path.name,
                "chunk_id": i,
                "content": text,
                "embedding": response["embedding"],
                "metadata": {}
            }).execute()

            time.sleep(1)

        print(f"   ✅ Done — {total} chunks ingested from {file_path.name}")

ingest_domain("pune")

In [ ]:
query = "What is special about chandrapur?"

query_embedding = genai.embed_content(
    model="models/gemini-embedding-2",
    content=query,
    task_type="retrieval_query",
    output_dimensionality=1536
)

results = supabase.rpc("match_documents", {
    "query_embedding": query_embedding["embedding"],
    "match_domain": "pune",
    "match_count": 5
}).execute()

for r in results.data:
    print(f"Similarity: {r['similarity']:.3f} | Source: {r['source']}")
    print(f"Content: {r['content'][:200]}")
    print("---")

In [ ]:
query = "What is special about Pune?"

query_embedding = genai.embed_content(
    model="models/gemini-embedding-2",
    content=query,
    task_type="retrieval_query",
    output_dimensionality=1536
)

results = supabase.rpc("match_documents", {
    "query_embedding": query_embedding["embedding"],
    "match_domain": "pune",
    "match_count": 5
}).execute()

for r in results.data:
    print(f"Similarity: {r['similarity']:.3f} | Source: {r['source']}")
    print(f"Content: {r['content'][:200]}")
    print("---")

In [ ]:
from typing import TypedDict, Optional

class RAGState(TypedDict):
    question: str           # original user question
    domain: str             # pune | mumbai | nagpur
    rewritten_question: str # rewritten version of question
    retrieved_chunks: list  # chunks returned from supabase
    grade: str              # good | bad
    retry_count: int        # how many times retrieval was retried
    web_search_results: str # results from serpapi if fallback
    final_answer: str       # final answer to return to user
    is_on_topic: bool       # classifier result

print("State defined successfully!")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY")
)

def classifier_node(state: RAGState) -> RAGState:
    question = state["question"]
    domain = state["domain"]

    prompt = f"""You are a strict domain classifier.
The user has selected the domain: {domain}
The user asked: {question}

The domain '{domain}' contains official documents, city development plans, 
infrastructure, economy, tourism and geographic information about {domain}.

Is this question genuinely relevant to the '{domain}' domain?
A question about food recipes, bollywood, cricket or anything 
not related to {domain} city information should be OFF-TOPIC.

Reply with only YES or NO."""

    response = llm.invoke(prompt)
    is_on_topic = response.content.strip().upper() == "YES"

    print(f"Classifier result: {'On-topic' if is_on_topic else 'Off-topic'}")
    return {**state, "is_on_topic": is_on_topic}

In [ ]:
def rewriter_node(state: RAGState) -> RAGState:
    question = state["question"]
    domain = state["domain"]

    prompt = f"""You are a query rewriter for a RAG system.
The user has selected the domain: {domain}
The original question is: {question}

Rewrite this question to be more specific and detailed 
to improve document retrieval. 
Return only the rewritten question, nothing else."""

    response = llm.invoke(prompt)
    rewritten = response.content.strip()

    print(f"Original   : {question}")
    print(f"Rewritten  : {rewritten}")

    return {**state, "rewritten_question": rewritten}


rewriter_node({
    "question": "What is special about Pune?",
    "domain": "pune",
    "rewritten_question": "",
    "retrieved_chunks": [],
    "grade": "",
    "retry_count": 0,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
})

In [ ]:
def retriever_node(state: RAGState) -> RAGState:
    question = state["rewritten_question"] or state["question"]
    domain = state["domain"]

    query_embedding = genai.embed_content(
        model="models/gemini-embedding-2",
        content=question,
        task_type="retrieval_query",
        output_dimensionality=1536
    )

    results = supabase.rpc("match_documents", {
        "query_embedding": query_embedding["embedding"],
        "match_domain": domain,
        "match_count": 5
    }).execute()

    chunks = results.data
    print(f"Retrieved {len(chunks)} chunks")
    for r in chunks:
        print(f"  Similarity: {r['similarity']:.3f} | {r['content'][:100]}")

    return {**state, "retrieved_chunks": chunks}


retriever_node({
    "question": "What is special about Pune?",
    "domain": "pune",
    "rewritten_question": "What are the unique cultural, historical, and culinary aspects that make Pune a special city in India?",
    "retrieved_chunks": [],
    "grade": "",
    "retry_count": 0,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
})

In [ ]:
def grader_node(state: RAGState) -> RAGState:
    question = state["rewritten_question"] or state["question"]
    chunks = state["retrieved_chunks"]
    retry_count = state["retry_count"]

    relevant_count = 0

    for chunk in chunks:
        prompt = f"""You are a retrieval grader.
Question: {question}
Retrieved chunk: {chunk['content']}

Is this chunk relevant to answer the question?
Reply with only YES or NO."""

        response = llm.invoke(prompt)
        if response.content.strip().upper() == "YES":
            relevant_count += 1

    total = len(chunks)
    relevance_ratio = relevant_count / total if total > 0 else 0
    grade = "good" if relevance_ratio >= 0.6 else "bad"

    print(f"Relevant chunks: {relevant_count}/{total}")
    print(f"Grade: {grade}")

    return {**state, "grade": grade, "retry_count": retry_count}


grader_node({
    "question": "What is special about Pune?",
    "domain": "pune",
    "rewritten_question": "What are the unique cultural, historical, and culinary aspects that make Pune a special city in India?",
    "retrieved_chunks": [{"content": "Pune is known for its rich history and culture.", "source": "Pune.pdf", "similarity": 0.75}],
    "grade": "",
    "retry_count": 0,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
})

In [ ]:
# first retrieve real chunks
state_after_retrieval = retriever_node({
    "question": "What is special about Pune?",
    "domain": "pune",
    "rewritten_question": "What are the unique cultural, historical, and culinary aspects that make Pune a special city in India?",
    "retrieved_chunks": [],
    "grade": "",
    "retry_count": 0,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
})

# now grade the real chunks
grader_node(state_after_retrieval)

In [ ]:
def grader_node(state: RAGState) -> RAGState:
    question = state["question"]  # use original question
    chunks = state["retrieved_chunks"]
    retry_count = state["retry_count"]

    relevant_count = 0

    for chunk in chunks:
        prompt = f"""You are a retrieval grader.
Question: {question}
Retrieved chunk: {chunk['content']}

Does this chunk contain useful information to answer the question?
Reply with only YES or NO."""

        response = llm.invoke(prompt)
        answer = response.content.strip().upper()
        print(f"  Chunk grade: {answer} | {chunk['content'][:80]}")
        if answer == "YES":
            relevant_count += 1

    total = len(chunks)
    relevance_ratio = relevant_count / total if total > 0 else 0
    grade = "good" if relevance_ratio >= 0.4 else "bad"

    print(f"\nRelevant chunks: {relevant_count}/{total}")
    print(f"Grade: {grade}")

    return {**state, "grade": grade, "retry_count": retry_count}

grader_node(state_after_retrieval)

In [ ]:
# start fresh with initial state
initial_state = {
    "question": "What is special about Pune?",
    "domain": "pune",
    "rewritten_question": "",
    "retrieved_chunks": [],
    "grade": "",
    "retry_count": 0,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
}

# chain all nodes
state1 = classifier_node(initial_state)
state2 = rewriter_node(state1)
state3 = retriever_node(state2)
state4 = grader_node(state3)

print("\nFinal grade:", state4["grade"])

In [ ]:
def refine_node(state: RAGState) -> RAGState:
    question = state["rewritten_question"] or state["question"]
    domain = state["domain"]
    retry_count = state["retry_count"] + 1

    prompt = f"""You are a query refinement expert.
The following question did not retrieve relevant results from the {domain} domain:
{question}

Rewrite it differently to improve retrieval. 
Be more specific, use different keywords.
Return only the refined question, nothing else."""

    response = llm.invoke(prompt)
    refined = response.content.strip()

    print(f"Retry count  : {retry_count}")
    print(f"Previous     : {question}")
    print(f"Refined      : {refined}")

    return {**state, "rewritten_question": refined, "retry_count": retry_count}


refine_node({
    "question": "What is special about Pune?",
    "domain": "pune",
    "rewritten_question": "What unique cultural aspects make Pune distinct?",
    "retrieved_chunks": [],
    "grade": "bad",
    "retry_count": 0,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
})

In [ ]:
from serpapi import GoogleSearch

def web_search_node(state: RAGState) -> RAGState:
    question = state["rewritten_question"] or state["question"]
    domain = state["domain"]

    print(f"Falling back to web search for: {question}")

    search = GoogleSearch({
        "q": f"{domain} {question}",
        "api_key": os.getenv("SERPAPI_KEY"),
        "num": 5
    })

    results = search.get_dict()
    organic = results.get("organic_results", [])

    web_results = "\n".join([
        f"- {r.get('title', '')}: {r.get('snippet', '')}"
        for r in organic
    ])

    print(f"Web results found: {len(organic)}")
    print(web_results[:300])

    return {**state, "web_search_results": web_results}


web_search_node({
    "question": "What is special about Pune?",
    "domain": "pune",
    "rewritten_question": "What unique cultural aspects make Pune distinct?",
    "retrieved_chunks": [],
    "grade": "bad",
    "retry_count": 3,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
})

In [ ]:
def generator_node(state: RAGState) -> RAGState:
    question = state["question"]
    chunks = state["retrieved_chunks"]
    web_results = state["web_search_results"]

    if chunks:
        context = "\n\n".join([
            f"Source: {c['source']}\n{c['content']}"
            for c in chunks
        ])
        source_type = "domain documents"
    else:
        context = web_results
        source_type = "web search"

    prompt = f"""You are a helpful assistant answering questions about a specific domain.
Question: {question}

Context from {source_type}:
{context}

Answer the question based only on the provided context.
Be clear and concise. If the context does not have enough information say so."""

    response = llm.invoke(prompt)
    answer = response.content.strip()

    print(f"Source used: {source_type}")
    print(f"\nAnswer:\n{answer}")

    return {**state, "final_answer": answer}


generator_node({
    **state4,
    "web_search_results": ""
})

In [ ]:
from langgraph.graph import StateGraph, END

def route_after_classifier(state: RAGState) -> str:
    if state["is_on_topic"]:
        return "rewriter"
    else:
        return END

def route_after_grader(state: RAGState) -> str:
    if state["grade"] == "good":
        return "generator"
    elif state["retry_count"] >= 3:
        return "web_search"
    else:
        return "refine"

def off_topic_node(state: RAGState) -> RAGState:
    message = f"Your question does not seem related to the '{state['domain']}' domain. Please ask a domain relevant question."
    print(message)
    return {**state, "final_answer": message}

# build graph
graph = StateGraph(RAGState)

# add all nodes
graph.add_node("classifier", classifier_node)
graph.add_node("rewriter", rewriter_node)
graph.add_node("retriever", retriever_node)
graph.add_node("grader", grader_node)
graph.add_node("refine", refine_node)
graph.add_node("web_search", web_search_node)
graph.add_node("generator", generator_node)
graph.add_node("off_topic", off_topic_node)

# set entry point
graph.set_entry_point("classifier")

# add edges
graph.add_conditional_edges("classifier", route_after_classifier, {
    "rewriter": "rewriter",
    END: "off_topic"
})
graph.add_edge("rewriter", "retriever")
graph.add_edge("retriever", "grader")
graph.add_conditional_edges("grader", route_after_grader, {
    "generator": "generator",
    "refine": "refine",
    "web_search": "web_search"
})
graph.add_edge("refine", "retriever")
graph.add_edge("web_search", "generator")
graph.add_edge("generator", END)
graph.add_edge("off_topic", END)

# compile
rag_app = graph.compile()

print("Graph compiled successfully!")

In [ ]:
from IPython.display import Image

Image(rag_app.get_graph().draw_mermaid_png())

In [ ]:
result = rag_app.invoke({
    "question": "What is special about Pune?",
    "domain": "pune",
    "rewritten_question": "",
    "retrieved_chunks": [],
    "grade": "",
    "retry_count": 0,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
})

print("\n====== FINAL ANSWER ======")
print(result["final_answer"])

In [ ]:
result = rag_app.invoke({
    "question": "What is special about Pune?",
    "domain": "pune",
    "rewritten_question": "",
    "retrieved_chunks": [],
    "grade": "",
    "retry_count": 0,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
})

print("\n====== FINAL ANSWER ======")
print(result["final_answer"])

In [ ]:
result = rag_app.invoke({
    "question": "What is the recipe for biryani?",
    "domain": "pune",
    "rewritten_question": "",
    "retrieved_chunks": [],
    "grade": "",
    "retry_count": 0,
    "web_search_results": "",
    "final_answer": "",
    "is_on_topic": False
})

print("\n====== FINAL ANSWER ======")
print(result["final_answer"])